# Stage 2: Triple-Barrier Labeling
This notebook implements triple-barrier labeling and dynamic volatility threshold.


In [ ]:
# ============================================================
# TICKER PARAMETER — change this to run for any stock
# ============================================================
TICKER = 'NVDA'   # Options: AAPL AMZN BAC GOOGL JNJ JPM MSFT NVDA UNH XOM

import os, sys
sys.path.insert(0, os.path.abspath('../..'))

DATA_RAW     = f'../../data/raw/{TICKER}_raw.csv'
DATA_CLEAN   = f'../../data/processed/per_stock/{TICKER}_clean.parquet'
DATA_CUSUM   = f'../../data/processed/per_stock/{TICKER}_cusum_events.parquet'
DATA_BARS    = f'../../data/processed/per_stock/{TICKER}_dollar_bars.parquet'
DATA_LABELS  = f'../../data/processed/per_stock/{TICKER}_labels.parquet'
DATA_WEIGHTS = f'../../data/processed/per_stock/{TICKER}_weights.parquet'
DATA_FRACDIFF= f'../../data/processed/per_stock/{TICKER}_fracdiff.parquet'
DATA_FEATURES= f'../../data/processed/per_stock/{TICKER}_ts_features.parquet'
DATA_BSADF   = f'../../data/processed/per_stock/{TICKER}_bsadf.parquet'
FIG_DIR      = f'../../reports/figures/per_stock/{TICKER}'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs('../../data/processed/per_stock', exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

sys.path.insert(0, os.path.abspath('../..'))
from src.labeling import get_daily_vol, add_vertical_barrier, apply_triple_barrier, get_bins, drop_labels

plt.style.use('seaborn-v0_8-darkgrid')


## 1. Load Data and CUSUM Events


In [ ]:
close_df = pd.read_parquet(f'../../data/processed/per_stock/{TICKER}_clean.parquet')
close = close_df['Adj Close']

cusum_events_idx = pd.read_parquet(f'../../data/processed/per_stock/{TICKER}_cusum_events.parquet').index
print(f"Loaded {len(cusum_events_idx)} CUSUM events.")


## 2. Compute Daily Volatility


In [ ]:
daily_vol = get_daily_vol(close, span=50)

# Create events dataframe with trgt
events = pd.DataFrame(index=cusum_events_idx)
events['trgt'] = daily_vol.loc[events.index]
events.dropna(subset=['trgt'], inplace=True)
print(f"Events after dropping NaNs in volatility: {len(events)}")


## 3. Set Vertical Barrier


In [ ]:
num_days = 10
t1 = add_vertical_barrier(close, events.index, num_days=num_days)
events['t1'] = t1
events.head()


## 4. Apply Triple Barrier


In [ ]:
pt_sl = [1.0, 1.0]
triple_barrier_events = apply_triple_barrier(close, events, pt_sl=pt_sl)
triple_barrier_events.head()


## 5. Get Labels


In [ ]:
labels = get_bins(triple_barrier_events, close)
# Drop rare labels (min_pct=0.05)
labels = drop_labels(labels, min_pct=0.05)
labels.head()


## 6. Label and Barrier Distribution


In [ ]:
print("Label Distribution:")
print(labels['bin'].value_counts())

print("\nBarrier Type Touched Distribution:")
triple_barrier_events_matched = triple_barrier_events.loc[labels.index]
barrier_type = pd.Series('t1', index=labels.index)
barrier_type.loc[labels['t1'] == triple_barrier_events_matched['pt']] = 'pt'
barrier_type.loc[labels['t1'] == triple_barrier_events_matched['sl']] = 'sl'
print(barrier_type.value_counts())


## 7. Plot Events on Price Chart


In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(close.index, close.values, label='NVDA Adj Close', color='black', alpha=0.5)

# Filter active dates for better visualization
start_date = labels.index.min() - pd.Timedelta(days=30)
ax.set_xlim(start_date, labels.index.max() + pd.Timedelta(days=30))

# Colors for labels: +1 green, -1 red, 0 grey
colors = {1.0: 'green', -1.0: 'red', 0.0: 'grey'}

for label_val, color in colors.items():
    idx = labels[labels['bin'] == label_val].index
    ax.scatter(idx, close.loc[idx], color=color, label=f'Label {int(label_val)}', s=20, zorder=5)

ax.set_yscale('log')
ax.set_title('Triple Barrier Labels on Price Chart')
ax.set_ylabel('Price (Log Scale)')
ax.legend()
plt.show()


## 8. Sensitivity Table


In [ ]:
pt_range = [0.5, 1.0, 1.5, 2.0]
sl_range = [0.5, 1.0, 1.5, 2.0]

results = []
for pt in pt_range:
    for sl in sl_range:
        tb_events = apply_triple_barrier(close, events, pt_sl=[pt, sl])
        lbls = get_bins(tb_events, close)
        vc = lbls['bin'].value_counts(normalize=True).to_dict()
        results.append({
            'pt': pt, 
            'sl': sl, 
            '-1': vc.get(-1.0, 0),
            '0': vc.get(0.0, 0),
            '1': vc.get(1.0, 0)
        })

sensitivity_df = pd.DataFrame(results).set_index(['pt', 'sl'])
display(sensitivity_df)


## 9. Save Labels


In [ ]:
# Merge ret and bin with the t1, pt, sl columns
final_labels = pd.concat([triple_barrier_events.loc[labels.index], labels[['ret', 'bin']]], axis=1)
final_labels.to_parquet(f'../../data/processed/per_stock/{TICKER}_labels.parquet')
print(f"Saved {TICKER} data to {DATA_CLEAN}")
